B21. Что именно добавляет синтез: таксономия эмерджентных пунктов.....


In [1]:
import sys, os, json, re

def _build_data_tools():
    import json
    import os
    import re
    import warnings
    import numpy as np
    import pandas as pd
    from scipy import stats
    warnings.filterwarnings('ignore')

    def path_join(*parts):
        return os.path.normpath(os.path.join(*parts))

    def normalize_newlines(frame):
        for column in frame.select_dtypes(include=['object', 'string']).columns:
            frame[column] = frame[column].map(lambda value: value.replace('\r\n', '\n').replace('\r', '\n') if isinstance(value, str) else value)
        return frame
    from pathlib import Path
    _start = Path.cwd().resolve()
    _repo = next((p for p in (_start, *_start.parents) if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()), None)
    if _repo is None:
        raise FileNotFoundError('Run the notebook from inside the repository.')
    if os.name == 'nt' and (not str(_repo).startswith(chr(92) * 2 + '?' + chr(92))):
        _repo = Path(chr(92) * 2 + '?' + chr(92) + str(_repo))
    ROOT = str(_repo)
    HERE = str(_repo / 'analyses/panel_mechanisms')
    OUT = path_join(HERE, 'results')
    os.makedirs(OUT, exist_ok=True)
    PBASE_DIR = 'experiments/clinical_protocols/delphi_panels/heterogeneous_gpt120_organizer_complete'
    HOMOG_DS_DIR = 'experiments/homogeneous_panels/three_deepseekv4_members_complete'
    NB = 'experiments/clinical_protocols/single_model_baselines/baseline_%s/llm_as_judge/baseline_%s_%s_judge.csv'
    USC = 'experiments/token_budget_controls/universal_self_consistency_gpt_seven_samples/llm_as_judge/usc_gpt_sc%d_judge.csv'
    HOMOG_DIRS = {'gpt': 'experiments/homogeneous_panels/three_gpt_members', 'qwen': 'experiments/homogeneous_panels/three_qwen_members', 'gemini': 'experiments/homogeneous_panels/three_gemini_members'}
    PANELS_OTHER_DIR = {'P-weak': 'experiments/clinical_protocols/delphi_panels/panel_qwen_organizer_gemini_deepseek_pro', 'P-mixed': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_gemini_glm', 'P-strong': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_glm_gpt120'}
    PANEL_FILE = {'P-weak': 'delphi_pweak_noroles_judge.csv', 'P-mixed': 'delphi_pmixed_noroles_judge.csv', 'P-strong': 'delphi_pstrong_noroles_judge.csv'}
    SINGLES = ['gpt', 'qwen', 'deepseekv4', 'deepseekv4pro', 'gemini', 'glm', 'kimi']
    COMPOSITION = {'P-base': ['gpt', 'qwen', 'deepseekv4'], 'P-weak': ['qwen', 'gemini', 'deepseekv4pro'], 'P-mixed': ['deepseekv4', 'gemini', 'glm'], 'P-strong': ['deepseekv4', 'glm', 'gpt'], 'homog gpt': ['gpt'] * 3, 'homog qwen': ['qwen'] * 3, 'homog deepseekv4': ['deepseekv4'] * 3, 'homog gemini': ['gemini'] * 3}
    HETERO = {'P-base': 1, 'P-weak': 1, 'P-mixed': 1, 'P-strong': 1, 'homog gpt': 0, 'homog qwen': 0, 'homog deepseekv4': 0, 'homog gemini': 0}
    PANELS = list(COMPOSITION)
    LADDER = ['baseline gpt', 'USC@3 gpt', 'USC@5 gpt', 'USC@7 gpt', 'homog gpt', 'homog deepseekv4', 'P-base']

    def arm_paths(design='noroles'):
        p = {}
        for m in SINGLES:
            p['baseline ' + m] = NB % (m, design, m)
        if design == 'noroles':
            for k in (3, 5, 7):
                p['USC@%d gpt' % k] = USC % k
            for tag, d in HOMOG_DIRS.items():
                p['homog ' + tag] = d + '/llm_as_judge/delphi_homog_%s_noroles_judge.csv' % tag
            p['homog deepseekv4'] = HOMOG_DS_DIR + '/llm_as_judge/delphi_homog_deepseekv4_noroles_judge.csv'
            for k, d in PANELS_OTHER_DIR.items():
                p[k] = d + '/llm_as_judge/' + PANEL_FILE[k]
        p['P-base'] = PBASE_DIR + '/llm_as_judge/delphi_base_%s_judge.csv' % design
        return p
    ROUNDS_DIR = {'P-base': (PBASE_DIR, 'delphi_base_noroles_protocol_r%d_judge.csv'), 'P-weak': (PANELS_OTHER_DIR['P-weak'], 'delphi_pweak_noroles_protocol_r%d_judge.csv'), 'P-mixed': (PANELS_OTHER_DIR['P-mixed'], 'delphi_pmixed_noroles_protocol_r%d_judge.csv'), 'P-strong': (PANELS_OTHER_DIR['P-strong'], 'delphi_pstrong_noroles_protocol_r%d_judge.csv'), 'homog gpt': (HOMOG_DIRS['gpt'], 'delphi_homog_gpt_noroles_protocol_r%d_judge.csv'), 'homog gemini': (HOMOG_DIRS['gemini'], 'delphi_homog_gemini_noroles_protocol_r%d_judge.csv'), 'homog deepseekv4': (HOMOG_DS_DIR, 'delphi_homog_deepseekv4_noroles_protocol_r%d_judge.csv')}
    RETEST = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_retest_r%d_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_retest_r%d_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_retest_r%d_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_retest_r%d_judge.csv'}
    SHORT = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_short_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_short_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_short_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_short_judge.csv'}
    JUDGE41 = {'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_gpt41/baseline_%s_gpt_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_gpt41/baseline_%s_qwen_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_gpt41/baseline_%s_deepseekv4_judge.csv'}
    CASE_COVARS = ['Type', 'Sub Clinical Specialties', 'Age Group', 'Genealogy', 'Diagnosis10', 'Vignette_race']
    DEXP = 'experiments/panel_parameter_sweeps'
    SWEEPS = {'experts': (DEXP + '/experts/llm_as_judge', 'delphi_experts_%s_judge.csv'), 'questionnaire_items': (DEXP + '/questionnaire_items/llm_as_judge', 'delphi_questionnaire_items_%s_judge.csv'), 'questions_per_expert': (DEXP + '/questions_per_expert/llm_as_judge', 'delphi_questions_per_expert_%s_judge.csv'), 'temperature': (DEXP + '/temperature/llm_as_judge', 'delphi_temperature_%s_judge.csv'), 'rounds': (DEXP + '/rounds/llm_as_judge', 'delphi_%s_judge.csv')}
    EXPERT_GROUPS = {'group1_10models': 10, 'group2_4models': 4, 'group3_5models': 5, 'group4_5models': 5, 'group5_7models': 7, 'group6_5models': 5, 'group7_6models': 6}
    EXT = 'experiments'
    MEDMCQA = {'baseline deepseekv4': EXT + '/external_benchmarks/medmcqa/baseline_deepseekv4/baseline_medmcqa_deepseekv4.csv', 'baseline gpt': EXT + '/external_benchmarks/medmcqa/baseline_gpt/baseline_medmcqa_gpt.csv', 'baseline qwen': EXT + '/external_benchmarks/medmcqa/baseline_qwen/baseline_medmcqa_qwen.csv', 'baseline alice': EXT + '/external_benchmarks/initial_runs/baseline_alice/baseline_medmcqa_alice.csv', 'baseline qwen36': EXT + '/external_benchmarks/initial_runs/baseline_qwen36/baseline_medmcqa_qwen36.csv', 'P-ac (gpt/qwen/ds)': EXT + '/external_benchmarks/medmcqa/panel_gpt120_organizer_qwen_deepseek/delphi_medmcqa_noroles.csv', 'P-ac (gpt/qwen/alice)': EXT + '/external_benchmarks/initial_runs/panel_gpt120_organizer_qwen_alice/delphi_medmcqa_noroles.csv', 'P-deepseektrio': EXT + '/external_benchmarks/initial_runs/three_deepseek_members/delphi_medmcqa_noroles.csv', 'P-base-like': EXT + '/external_benchmarks/initial_runs/heterogeneous_gpt120_organizer/delphi_medmcqa_noroles.csv'}

    def rd_any(rel, index=None):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        with open(p, encoding='utf-8-sig', errors='replace') as fh:
            head = fh.readline()
        sep = ';' if head.count(';') > head.count(',') else ','
        d = normalize_newlines(pd.read_csv(p, sep=sep, encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if index and index in d.columns:
            d = d.dropna(subset=[index]).drop_duplicates(index).set_index(index)
        return d

    def sweep_arms(kind, verbose=True):
        import glob as _g
        folder, _ = SWEEPS[kind]
        out = {}
        for p in sorted(_g.glob(path_join(ROOT, folder, '*_judge.csv'))):
            name = os.path.basename(p)[:-len('_judge.csv')]
            d = rd(os.path.relpath(p, ROOT).replace('\\', '/'))
            if d is not None:
                out[name] = d
        if verbose:
            print('свип %s: плеч %d, кейсов %d' % (kind, len(out), len(next(iter(out.values()))) if out else 0))
        return out

    def member_triage_acc(d):
        t = true_label(d)
        out = {}
        for p in participants(d):
            lab = d['r1_emergency__' + p].map(norm_label)
            ok = lab.notna() & t.notna()
            out[p] = float((lab[ok] == t[ok]).mean()) if ok.any() else np.nan
        return pd.Series(out)

    def rd(rel, need_score=True):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        d = normalize_newlines(pd.read_csv(p, sep=';', encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if need_score and 'Judge_score' not in d.columns:
            return None
        sub = ['Judge_score', 'Vignette'] if need_score else ['Vignette']
        d = d.dropna(subset=sub).drop_duplicates('Vignette', keep='first')
        return d.set_index('Vignette')

    def load_arms(design='noroles', verbose=True):
        a = {}
        for name, rel in arm_paths(design).items():
            d = rd(rel)
            if d is not None:
                a[name] = d
        if verbose:
            print('плеч загружено: %d' % len(a))
            print('   ' + ', '.join(a))
        return a

    def js(arms, name):
        return pd.to_numeric(arms[name]['Judge_score'], errors='coerce').dropna()

    def score_matrix(arms, names=None):
        names = list(arms) if names is None else names
        m = pd.concat([js(arms, n).rename(n) for n in names], axis=1)
        return m.dropna()

    def meta(arms=None):
        d = rd(PBASE_DIR + '/llm_as_judge/delphi_base_noroles_judge.csv')
        cols = [c for c in CASE_COVARS if c in d.columns]
        return d[cols].copy()

    def item_frame(d):
        r = {}
        if d is None or 'judge_details' not in d.columns:
            return r
        for v, s in d['judge_details'].dropna().items():
            try:
                arr = json.loads(s)
            except Exception:
                continue
            for it in arr:
                ax = next((t.split(':', 1)[1] for t in it.get('tags') or [] if isinstance(t, str) and t.startswith('axis:')), None)
                r[v, it['id']] = {'met': bool(it.get('met')), 'points': it.get('points', 0) or 0, 'axis': ax}
        return r

    def load_items(design='noroles', arms_wanted=None, verbose=True):
        paths = arm_paths(design)
        if arms_wanted:
            paths = {k: v for k, v in paths.items() if k in arms_wanted}
        items = {}
        for name, rel in paths.items():
            d = rd(rel)
            f = item_frame(d)
            if f:
                items[name] = f
        common = None
        for f in items.values():
            common = set(f) if common is None else common & set(f)
        ref = next(iter(items.values()))
        pos = sorted([k for k in common or set() if ref[k]['points'] > 0])
        if verbose:
            print('плеч с попунктными вердиктами: %d' % len(items))
            print('общих положительных пунктов: %d' % len(pos))
        return (items, pos)

    def items_long(items, pos, arms_wanted=None):
        names = arms_wanted or list(items)
        rows = []
        for a in names:
            f = items[a]
            for k in pos:
                it = f[k]
                rows.append({'case': k[0], 'cid': k[1], 'arm': a, 'met': int(it['met']), 'points': it['points'], 'axis': it['axis']})
        return pd.DataFrame(rows)

    def paired(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(float), b.loc[ix].astype(float))
        d = x - y
        p = stats.wilcoxon(x, y)[1] if (d != 0).any() else np.nan
        t = stats.ttest_rel(x, y)
        return {'n': len(ix), 'delta': float(d.mean()), 'sd': float(d.std(ddof=1)), 'share_gt': float((d > 0).mean()), 'p_wilcoxon': float(p), 't': float(t.statistic), 'p_ttest': float(t.pvalue)}

    def se_clt(x):
        x = np.asarray(x, float)
        n = len(x)
        return float(np.sqrt(x.var(ddof=1) / n))

    def se_clustered(x, cluster, cr1=True):
        x = np.asarray(x, float)
        cl = np.asarray(cluster)
        n = len(x)
        e = x - x.mean()
        df = pd.DataFrame({'e': e, 'c': cl})
        r = df.groupby('c', observed=True)['e'].sum().to_numpy()
        ss = df.groupby('c', observed=True)['e'].apply(lambda s: float((s ** 2).sum())).to_numpy()
        cross = float((r ** 2 - ss).sum())
        v = se_clt(x) ** 2 + cross / n ** 2
        G = len(r)
        if cr1 and G > 1:
            v *= G / (G - 1.0)
        return (float(np.sqrt(max(v, 0.0))), G)

    def cluster_test(d, cluster, cr1=True):
        d = pd.Series(d).astype(float)
        cl = pd.Series(cluster).reindex(d.index)
        ok = cl.notna()
        d, cl = (d[ok], cl[ok])
        m = float(d.mean())
        s0 = se_clt(d)
        s1, G = se_clustered(d, cl, cr1=cr1)
        dfree = max(G - 1, 1)
        t = m / s1 if s1 > 0 else np.nan
        p = float(2 * stats.t.sf(abs(t), dfree)) if np.isfinite(t) else np.nan
        tc = stats.t.ppf(0.975, dfree)
        return {'n': len(d), 'G': G, 'mean': m, 'se_naive': s0, 'se_cluster': s1, 'DEFF': (s1 / s0) ** 2 if s0 > 0 else np.nan, 't': float(t), 'df': dfree, 'p': p, 'ci_lo': m - tc * s1, 'ci_hi': m + tc * s1, 'p_naive': float(2 * stats.norm.sf(abs(m / s0))) if s0 > 0 else np.nan}

    def icc_anova(values, groups):
        df = pd.DataFrame({'y': np.asarray(values, float), 'g': np.asarray(groups)})
        df = df.dropna()
        grp = df.groupby('g', observed=True)['y']
        ni = grp.size().to_numpy().astype(float)
        k = len(ni)
        N = ni.sum()
        if k < 2 or N <= k:
            return (0.0, float(ni.mean() if k else 1), k)
        gm = df['y'].mean()
        msb = float((ni * (grp.mean().to_numpy() - gm) ** 2).sum() / (k - 1))
        ssw = float(((df['y'] - df['g'].map(grp.mean()).to_numpy()) ** 2).sum())
        msw = ssw / (N - k)
        B0 = float((N - (ni ** 2).sum() / N) / (k - 1))
        rho = (msb - msw) / (msb + (B0 - 1) * msw) if msb + (B0 - 1) * msw > 0 else 0.0
        return (float(np.clip(rho, 0, 1)), B0, k)

    def ci_wald(p, n, z=1.96):
        se = np.sqrt(p * (1 - p) / n)
        return (p - z * se, p + z * se)

    def ci_wilson(p, n, z=1.96):
        d = 1 + z ** 2 / n
        c = (p + z ** 2 / (2 * n)) / d
        h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / d
        return (c - h, c + h)

    def ci_agresti_coull(x, n, z=1.96):
        nt = n + z ** 2
        pt = (x + z ** 2 / 2) / nt
        se = np.sqrt(pt * (1 - pt) / nt)
        return (pt - z * se, pt + z * se)

    def ci_clopper_pearson(x, n, alpha=0.05):
        lo = 0.0 if x == 0 else stats.beta.ppf(alpha / 2, x, n - x + 1)
        hi = 1.0 if x == n else stats.beta.ppf(1 - alpha / 2, x + 1, n - x)
        return (float(lo), float(hi))

    def boot_ci(x, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        x = np.asarray(x, float)
        n = len(x)
        s = np.array([stat(x[rng.integers(0, n, n)]) for _ in range(B)])
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def boot_ci_cluster(x, cluster, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        df = pd.DataFrame({'x': np.asarray(x, float), 'c': np.asarray(cluster)})
        groups = [g['x'].to_numpy() for _, g in df.groupby('c', observed=True)]
        G = len(groups)
        s = []
        for _ in range(B):
            pick = rng.integers(0, G, G)
            s.append(stat(np.concatenate([groups[i] for i in pick])))
        s = np.array(s)
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def holm(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 0.0
        for i, k in enumerate(order):
            run = max(run, (m - i) * p[k])
            adj[k] = min(run, 1.0)
        return adj

    def bh(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 1.0
        for i in range(m - 1, -1, -1):
            k = order[i]
            run = min(run, m * p[k] / (i + 1))
            adj[k] = min(run, 1.0)
        return adj

    def bonferroni(pvals):
        p = np.asarray(pvals, float)
        return np.minimum(p * len(p), 1.0)

    def mcnemar(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(bool), b.loc[ix].astype(bool))
        n01, n10 = (int((~x & y).sum()), int((x & ~y).sum()))
        if n01 + n10 == 0:
            return {'n01': n01, 'n10': n10, 'p': 1.0}
        return {'n01': n01, 'n10': n10, 'p': float(stats.binomtest(n10, n01 + n10, 0.5).pvalue)}
    ORD = {'SELF': 0, 'NONEMERG': 1, 'EMERG': 2}
    GT = {'Self-care appropriate': 'SELF', 'Requires non-emergent care': 'NONEMERG', 'Requires emergent care': 'EMERG'}
    _DASH = dict.fromkeys(map(ord, '‐‑‒–—―−\xad'), '-')
    _KEYS = [('requires emergent care', 'EMERG'), ('requires non-emergent care', 'NONEMERG'), ('self-care appropriate', 'SELF')]

    def norm_label(x):
        if not isinstance(x, str) or not x.strip():
            return None
        if x.strip() in GT:
            return GT[x.strip()]
        import unicodedata
        s = unicodedata.normalize('NFKC', x).translate(_DASH).lower()
        s = re.sub('\\s+', ' ', re.sub('[^a-z\\- ]', ' ', s)).strip()
        for k, v in _KEYS:
            if s.startswith(k) or k in s:
                return v
        if 'non-emergent' in s or 'non emergent' in s:
            return 'NONEMERG'
        if 'emergent' in s or 'emergency' in s:
            return 'EMERG'
        if 'self' in s:
            return 'SELF'
        return None
    _PAT = re.compile('emergency\\s*type\\s*[:\\-]\\s*[\\*_`#\\s]*([^\\n\\r\\*_|]{0,90})', re.I)

    def parse_triage(text):
        if not isinstance(text, str) or not text.strip():
            return None
        import unicodedata
        m = _PAT.search(unicodedata.normalize('NFKC', text).translate(_DASH))
        if not m:
            return None
        return norm_label('Requires ' + m.group(1) if False else m.group(1))

    def pred_triage(d):
        if 'delphi_consensus_emergency_type' in d.columns:
            s = d['delphi_consensus_emergency_type'].map(norm_label)
            if s.notna().mean() > 0.5:
                return s
        for c in ('winner_protocol', 'baseline_protocol_anonymous'):
            if c in d.columns:
                s = d[c].map(parse_triage)
                if s.notna().mean() > 0.3:
                    return s
        return pd.Series(index=d.index, dtype=object)

    def participants(d):
        return sorted((c.split('__', 1)[1] for c in d.columns if c.startswith('r1_emergency__')))

    def true_label(d):
        return d['Type'].map(GT)

    def words(s):
        return len(re.findall('\\S+', str(s))) if isinstance(s, str) else np.nan

    def fmt_p(p):
        if p is None or (isinstance(p, float) and (not np.isfinite(p))):
            return '  n/a '
        return '%.3f' % p if p >= 0.001 else '%.1e' % p

    def show(df, title=None, floatfmt='%.4f'):
        if title:
            print('\n' + title)
            print('-' * max(len(title), 60))
        with pd.option_context('display.width', 200, 'display.max_columns', 60, 'display.float_format', lambda v: floatfmt % v):
            print(df)

    def save(df, name):
        p = path_join(OUT, name)
        df.to_csv(p, sep=';', encoding='utf-8-sig')
        print('сохранено: out/%s  (%d строк)' % (name, len(df)))
        return p
    return locals()
globals().update({k: v for k, v in _build_data_tools().items() if not k.startswith('_')})
del _build_data_tools
import statsmodels.api as sm
A = load_arms()
ITEMS, POS = load_items()
print()
AX_ORDER = ['accuracy', 'completeness', 'reasoning', 'urgency', 'uncertainty', 'communication_quality']


плеч загружено: 18
   baseline gpt, baseline qwen, baseline deepseekv4, baseline deepseekv4pro, baseline gemini, baseline glm, baseline kimi, USC@3 gpt, USC@5 gpt, USC@7 gpt, homog gpt, homog qwen, homog gemini, homog deepseekv4, P-weak, P-mixed, P-strong, P-base


плеч с попунктными вердиктами: 18
общих положительных пунктов: 2249



In [2]:
def classify(pname):
    comp = COMPOSITION[pname]
    members = ['baseline ' + m for m in comp]
    if pname not in ITEMS or not all((m in ITEMS for m in members)):
        return None
    rows = []
    for k in POS:
        v = [int(ITEMS[m][k]['met']) for m in members]
        pn = int(ITEMS[pname][k]['met'])
        s = sum(v)
        cls = 'эмерджентный' if pn == 1 and s == 0 else 'потерянный' if pn == 0 and s >= 2 else 'подхвачен у меньшинства' if pn == 1 and s == 1 else 'согласие'
        rows.append({'case': k[0], 'cid': k[1], 'класс': cls, 'panel': pn, 'members_met': s, 'points': ITEMS[pname][k]['points'], 'axis': ITEMS[pname][k]['axis']})
    return pd.DataFrame(rows)
CL = {p: classify(p) for p in PANELS if p in A}
CL = {k: v for k, v in CL.items() if v is not None}
rows = []
for p, D in CL.items():
    vc = D['класс'].value_counts()
    n = len(D)
    rows.append({'панель': p, 'пунктов': n, 'эмерджентных': int(vc.get('эмерджентный', 0)), 'потерянных': int(vc.get('потерянный', 0)), 'подхвачено у меньшинства': int(vc.get('подхвачен у меньшинства', 0)), 'доля эмерджентных': vc.get('эмерджентный', 0) / n, 'доля потерянных': vc.get('потерянный', 0) / n, 'нетто': int(vc.get('эмерджентный', 0)) - int(vc.get('потерянный', 0))})
S = pd.DataFrame(rows).set_index('панель')
show(S, 'Эмерджентные и потерянные пункты по панелям')
print()
print('«подхвачен у меньшинства» — пункт выполнил ровно один участник, и синтез')
print('его сохранил. Это НЕ эмерджентность, а корректная работа отбора, но')
print('важная: она показывает, что организатор не голосует большинством.')



Эмерджентные и потерянные пункты по панелям
------------------------------------------------------------
                  пунктов  эмерджентных  потерянных  подхвачено у меньшинства  доля эмерджентных  доля потерянных  нетто
панель                                                                                                                  
P-base               2249           246          69                       236             0.1094           0.0307    177
P-weak               2249           319          36                       249             0.1418           0.0160    283
P-mixed              2249           311          36                       264             0.1383           0.0160    275
P-strong             2249           265          60                       244             0.1178           0.0267    205
homog gpt            2249           428         105                         0             0.1903           0.0467    323
homog qwen           2249           401        

In [3]:
rows = []
for p, D in CL.items():
    tot = D.groupby('axis').size()
    em = D[D['класс'] == 'эмерджентный'].groupby('axis').size()
    lo = D[D['класс'] == 'потерянный'].groupby('axis').size()
    for ax in tot.index:
        rows.append({'панель': p, 'ось': ax, 'пунктов': int(tot[ax]), 'эмерджентных': int(em.get(ax, 0)), 'потерянных': int(lo.get(ax, 0)), 'доля эмерджентных': em.get(ax, 0) / tot[ax], 'доля потерянных': lo.get(ax, 0) / tot[ax]})
AXD = pd.DataFrame(rows)
piv = AXD.pivot_table(index='ось', values=['доля эмерджентных', 'доля потерянных'], aggfunc='mean')
piv['нетто'] = piv['доля эмерджентных'] - piv['доля потерянных']
piv = piv.reindex([a for a in AX_ORDER if a in piv.index])
show(piv, 'Эмерджентность и потери по осям рубрики (среднее по панелям)')
print()
print('Проверка однородности: одинакова ли доля эмерджентных пунктов по осям?')
for p, D in CL.items():
    t = pd.crosstab(D['axis'], D['класс'] == 'эмерджентный')
    if t.shape == (len(t), 2) and t.values.min() >= 0:
        chi2, pv, dof, _ = stats.chi2_contingency(t)
        print('   %-18s chi2 = %7.2f  df = %d  p = %s' % (p, chi2, dof, fmt_p(pv)))



Эмерджентность и потери по осям рубрики (среднее по панелям)
------------------------------------------------------------
                       доля потерянных  доля эмерджентных  нетто
ось                                                             
accuracy                        0.0259             0.0914 0.0655
completeness                    0.0384             0.2118 0.1733
reasoning                       0.0144             0.1287 0.1143
urgency                         0.0284             0.1449 0.1165
uncertainty                     0.0144             0.2778 0.2634
communication_quality           0.0402             0.1508 0.1106

Проверка однородности: одинакова ли доля эмерджентных пунктов по осям?
   P-base             chi2 =   90.55  df = 5  p = 5.1e-18
   P-weak             chi2 =   49.65  df = 5  p = 1.6e-09
   P-mixed            chi2 =   77.51  df = 5  p = 2.8e-15
   P-strong           chi2 =  120.65  df = 5  p = 2.3e-24
   homog gpt          chi2 =  100.06  df = 5  p = 5.1

In [4]:
SINGLE_ITEMS = {m: ITEMS[m] for m in ITEMS if m.startswith('baseline ')}
rows = []
for p, D in CL.items():
    comp = set(('baseline ' + m for m in COMPOSITION[p]))
    outside = [m for m in SINGLE_ITEMS if m not in comp]
    if len(outside) < 2:
        continue
    diff = {k: 1 - np.mean([ITEMS[m][k]['met'] for m in outside]) for k in POS}
    D = D.copy()
    D['трудность'] = [diff[r.case, r.cid] for r in D.itertuples()]
    D['страта'] = pd.cut(D['трудность'], [-0.001, 0.001, 0.25, 0.5, 0.75, 1.001], labels=['очень лёгкие', 'лёгкие', 'средние', 'трудные', 'очень трудные'])
    g = D.groupby('страта', observed=True).agg(пунктов=('класс', 'size'), эмерджентных=('класс', lambda s: int((s == 'эмерджентный').sum())), потерянных=('класс', lambda s: int((s == 'потерянный').sum())))
    g['доля эмерджентных'] = g['эмерджентных'] / g['пунктов']
    g['доля потерянных'] = g['потерянных'] / g['пунктов']
    g['панель'] = p
    rows.append(g.reset_index())
DIFF = pd.concat(rows, ignore_index=True)
show(DIFF.pivot_table(index='страта', values=['доля эмерджентных', 'доля потерянных'], aggfunc='mean', observed=True), 'Эмерджентность и потери по стратам трудности (среднее по панелям)')
print()
print('Ожидание из H3.7: потери должны концентрироваться на ЛЁГКИХ пунктах')
print('(синтез роняет тривиальное), а эмерджентность — на ТРУДНЫХ.')



Эмерджентность и потери по стратам трудности (среднее по панелям)
-----------------------------------------------------------------
               доля потерянных  доля эмерджентных
страта                                           
очень лёгкие            0.0096             0.0252
лёгкие                  0.0311             0.1314
средние                 0.0557             0.2109
трудные                 0.0616             0.2896
очень трудные           0.0246             0.2737

Ожидание из H3.7: потери должны концентрироваться на ЛЁГКИХ пунктах
(синтез роняет тривиальное), а эмерджентность — на ТРУДНЫХ.


In [5]:
PANEL_REL = {'P-base': PBASE_DIR + '/llm_as_judge/delphi_base_noroles_judge.csv', 'P-weak': PANELS_OTHER_DIR['P-weak'] + '/llm_as_judge/' + PANEL_FILE['P-weak'], 'P-mixed': PANELS_OTHER_DIR['P-mixed'] + '/llm_as_judge/' + PANEL_FILE['P-mixed'], 'P-strong': PANELS_OTHER_DIR['P-strong'] + '/llm_as_judge/' + PANEL_FILE['P-strong'], 'homog gpt': HOMOG_DIRS['gpt'] + '/llm_as_judge/delphi_homog_gpt_noroles_judge.csv', 'homog deepseekv4': HOMOG_DS_DIR + '/llm_as_judge/delphi_homog_deepseekv4_noroles_judge.csv'}
rows = []
for p, D in CL.items():
    if p not in PANEL_REL:
        continue
    d = rd(PANEL_REL[p])
    w = d['winner_protocol'].map(words)
    per = D.groupby('case').agg(эмерджентных=('класс', lambda s: int((s == 'эмерджентный').sum())), потерянных=('класс', lambda s: int((s == 'потерянный').sum())), пунктов=('класс', 'size'))
    per['слов'] = w.reindex(per.index)
    per = per.dropna()
    if len(per) < 30:
        continue
    r1, p1 = stats.spearmanr(per['слов'], per['эмерджентных'])
    r2, p2 = stats.spearmanr(per['слов'], per['потерянных'])
    rows.append({'панель': p, 'кейсов': len(per), 'rho (длина, эмерджентные)': r1, 'p': p1, 'rho (длина, потерянные)': r2, 'p ': p2, 'ср. эмерджентных на кейс': float(per['эмерджентных'].mean())})
show(pd.DataFrame(rows).set_index('панель'), 'Скептическая проверка: длина синтеза против эмерджентности')
print()
print('Если rho для эмерджентных высока и значима, версия «дело в длине»')
print('получает поддержку, и заявлять эмерджентность надо осторожнее.')
print('Если rho мала, эмерджентность не сводится к многословию.')



Скептическая проверка: длина синтеза против эмерджентности
------------------------------------------------------------
                  кейсов  rho (длина, эмерджентные)      p  rho (длина, потерянные)     p   ср. эмерджентных на кейс
панель                                                                                                              
P-base               220                     0.0179 0.7916                   0.0164 0.8085                    1.1182
P-weak               220                    -0.0717 0.2897                  -0.0379 0.5758                    1.4500
P-mixed              220                     0.2752 0.0000                  -0.1547 0.0217                    1.4136
P-strong             220                     0.2685 0.0001                  -0.1517 0.0245                    1.2045
homog gpt            220                     0.0752 0.2670                  -0.0435 0.5210                    1.9455
homog deepseekv4     220                     0.0560 0.4088  

In [6]:
rows = []
for p, D in CL.items():
    em = D[D['класс'] == 'эмерджентный']['points']
    lo = D[D['класс'] == 'потерянный']['points']
    al = D['points']
    rows.append({'панель': p, 'ср. вес эмерджентных': float(em.mean()) if len(em) else np.nan, 'ср. вес потерянных': float(lo.mean()) if len(lo) else np.nan, 'ср. вес всех': float(al.mean()), 'нетто по штукам': len(em) - len(lo), 'нетто по баллам': float(em.sum() - lo.sum())})
show(pd.DataFrame(rows).set_index('панель'), 'Вес эмерджентных и потерянных пунктов')



Вес эмерджентных и потерянных пунктов
------------------------------------------------------------
                  ср. вес эмерджентных  ср. вес потерянных  ср. вес всех  нетто по штукам  нетто по баллам
панель                                                                                                    
P-base                          5.6301              6.3333        6.5434              177         948.0000
P-weak                          5.8558              6.3333        6.5434              283        1640.0000
P-mixed                         5.7974              6.1389        6.5434              275        1582.0000
P-strong                        5.7396              6.1333        6.5434              205        1153.0000
homog gpt                       5.7687              6.0667        6.5434              323        1832.0000
homog qwen                      6.0773              6.1429        6.5434              303        1835.0000
homog deepseekv4                5.8855      

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

def criterion_texts(d):
    out = {}
    for v, blob in d['Criterion'].dropna().items():
        parts = re.split('(?m)^\\[(\\d+)\\]\\n', str(blob).replace('\r\n', '\n'))
        for k in range(1, len(parts), 2):
            m = re.match('points:\\s*\\S+\\ntags:[^\\n]*\\ncriterion:\\n(.*)$', parts[k + 1], re.S)
            if m:
                out[v, int(parts[k])] = m.group(1).strip()
    return out
TX = criterion_texts(rd(PANEL_REL['P-base']))
print('текстов критериев извлечено:', len(TX))
D = CL['P-base'].copy()
D['text'] = [TX.get((r.case, r.cid), '') for r in D.itertuples()]
D = D[D['text'].str.len() > 0]
groups = {'эмерджентные': D[D['класс'] == 'эмерджентный']['text'], 'потерянные': D[D['класс'] == 'потерянный']['text'], 'остальные': D[~D['класс'].isin(['эмерджентный', 'потерянный'])]['text']}
docs = [' '.join(v.tolist()) for v in groups.values()]
vec = TfidfVectorizer(max_features=4000, ngram_range=(1, 2), min_df=1, stop_words='english')
X = vec.fit_transform(docs).toarray()
terms = np.array(vec.get_feature_names_out())
for i, name in enumerate(groups):
    top = np.argsort(-X[i])[:18]
    print('\n%-14s (n = %d пунктов)' % (name, len(groups[name])))
    print('   ', ', '.join(terms[top]))


текстов критериев извлечено: 3269



эмерджентные   (n = 246 пунктов)
    risk, recommends, differential, considers, given, mentions, notes, appropriate, discusses, management, history, addresses, disease, asks, prevention, including, care, provides

потерянные     (n = 69 пунктов)
    recommends, notes, severe, treatment, needed, explains, symptoms, antibiotics, mentions, patient, signs, viral, given, risk, antibiotic, use, fever, pain

остальные      (n = 1934 пунктов)
    recommends, identifies, notes, based, pain, patient, therapy, care, symptoms, given, risk, considers, provides, diagnosis, likely, management, discusses, severe


In [8]:
ex = D[D['класс'] == 'эмерджентный'].sort_values('points', ascending=False).head(8)
for i, r in enumerate(ex.itertuples(), 1):
    print('%d. [вес %s, ось %s]' % (i, r.points, r.axis))
    print('   %s' % re.sub('\\s+', ' ', r.text)[:300])
    print()


1. [вес 9, ось urgency]
   Explicitly warns to monitor for tension pneumothorax (sudden hemodynamic collapse, tracheal deviation, distended neck veins) requiring immediate needle decompression.

2. [вес 9, ось completeness]
   Orders objective severity assessment: peak expiratory flow rate (or FEV1) compared to personal best/predicted, pulse oximetry, vital signs, and assessment of accessory muscle use, ability to speak in full sentences, and mental status.

3. [вес 9, ось uncertainty]
   Identifies the differential beyond uncomplicated mechanical strain: L4-L5 disc herniation with L5 radiculopathy, peroneal nerve compression, cauda equina syndrome, spinal mass, epidural abscess.

4. [вес 8, ось accuracy]
   Recommends supportive care with adequate hydration, weight-appropriate acetaminophen or ibuprofen for pain or fever, and cool soft foods, while avoiding potentially unsafe topical oral anesthetics

5. [вес 8, ось completeness]
   Recommends referral to gastroenterology for upper en

In [9]:
save(S.reset_index(), 'emergent_criterion_recovery_counts.csv')
save(AXD, 'emergent_criterion_recovery_by_axis.csv')
save(DIFF, 'emergent_criterion_recovery_by_difficulty.csv')
save(pd.concat([v.assign(панель=k) for k, v in CL.items()]), 'emergent_criterion_recovery_items.csv')


сохранено: out/emergent_criterion_recovery_counts.csv  (8 строк)
сохранено: out/emergent_criterion_recovery_by_axis.csv  (48 строк)
сохранено: out/emergent_criterion_recovery_by_difficulty.csv  (40 строк)


сохранено: out/emergent_criterion_recovery_items.csv  (17992 строк)


'\\\\?\\D:\\research_llmfmed\\research_llmfmed\\1design_alpha\\dataverse_gold\\github_repository\\analyses\\panel_mechanisms\\results\\emergent_criterion_recovery_items.csv'